# Lab 10: Text Summarisation and Question Answering with Pretrained LLMs

This notebook builds a small text summarisation and question-answering application using pretrained transformer language models from Hugging Face.

No model training is required. The application lets a user:

1. Enter a paragraph.
2. Generate a summary of that paragraph.
3. Ask multiple questions based on the paragraph.
4. Type `quit` to stop asking questions.

## Pretrained Models Used

The following pretrained models are used from the web via the Hugging Face model hub:

- **Summarisation:** [`facebook/bart-large-cnn`](https://huggingface.co/facebook/bart-large-cnn)  
  A BART-large model pretrained and fine-tuned for CNN/Daily Mail style abstractive summarisation.

- **Question Answering:** [`deepset/roberta-base-squad2`](https://huggingface.co/deepset/roberta-base-squad2)  
  A RoBERTa model fine-tuned on SQuAD2 for extractive question answering.

Both models are loaded through the `transformers` pipeline API, so they can be used directly without training.

In [1]:
# Install dependencies if they are not already available.
# Run this cell once before loading the pretrained models.
import sys
import subprocess
import importlib.util

required_packages = ["transformers", "torch", "sentencepiece"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")

All required packages are already installed.


In [2]:
from transformers import pipeline

SUMMARISATION_MODEL = "facebook/bart-large-cnn"
QA_MODEL = "deepset/roberta-base-squad2"

print("Loading summarisation model...")
summariser = pipeline("summarization", model=SUMMARISATION_MODEL)

print("Loading question-answering model...")
qa_pipeline = pipeline("question-answering", model=QA_MODEL)

print("Models loaded successfully.")

c:\Users\nguye\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading summarisation model...



Device set to use cpu


Loading question-answering model...


Fetching 0 files: 0it [00:00, ?it/s]
Fetching 1 files: 100%|██████████| 1/1 [00:00<?, ?it/s]
Fetching 0 files: 0it [00:00, ?it/s]
Device set to use cpu


Models loaded successfully.


In [3]:
def summarise_paragraph(paragraph, max_length=90, min_length=30):
    """Return a concise summary for the input paragraph."""
    if not paragraph or not paragraph.strip():
        return "Please enter a non-empty paragraph."

    result = summariser(
        paragraph,
        max_length=max_length,
        min_length=min_length,
        do_sample=False,
        truncation=True,
    )
    return result[0]["summary_text"]


def answer_question(paragraph, question):
    """Answer a question using only the provided paragraph as context."""
    if not paragraph or not paragraph.strip():
        return "Please enter a paragraph first."

    if not question or not question.strip():
        return "Please enter a non-empty question."

    result = qa_pipeline(question=question, context=paragraph)
    return result["answer"]


def run_text_summarisation_qa_app():
    """Interactive console application for summarisation and question answering."""
    print("Text Summarisation and Question Answering Application")
    print("Type 'quit' at any prompt to exit.\n")

    paragraph = input("Enter paragraph: ").strip()
    if paragraph.lower() == "quit":
        print("Goodbye.")
        return

    if not paragraph:
        print("No paragraph entered. Please run the application again.")
        return

    summary = summarise_paragraph(paragraph)
    print("\nSummary:")
    print(summary)

    while True:
        question = input("\nEnter question based on the paragraph, or type 'quit': ").strip()

        if question.lower() == "quit":
            print("Goodbye.")
            break

        if not question:
            print("Please enter a question or type 'quit'.")
            continue

        answer = answer_question(paragraph, question)
        print("Answer:", answer)

## Run the Interactive Application

Run the next cell to start the application. Example prompts:

- `Enter paragraph:`
- `Enter question based on the paragraph, or type 'quit':`

You can ask multiple questions about the same paragraph before typing `quit`.

In [5]:
run_text_summarisation_qa_app()

Text Summarisation and Question Answering Application
Type 'quit' at any prompt to exit.



Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Your max_length is set to 90, but your input_length is only 6. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)



Summary:
CNN.com will feature iReporter photos in a weekly Travel Snapshots gallery. Visit CNN.com/Travel each week for a new gallery of snapshots.
Answer: How are you
Goodbye.


## Sample Test Inputs

The next cell tests the application functions without using keyboard input. This makes it easier to verify that summarisation and question answering work correctly.

In [6]:
sample_paragraph = """
Artificial intelligence is transforming many industries by helping organisations automate tasks,
analyse large amounts of data, and support decision making. In healthcare, AI systems can assist
with medical image analysis, patient monitoring, and drug discovery. In education, AI tools can
support personalised learning by adapting content to each student's needs. However, the use of AI
also raises concerns about privacy, fairness, transparency, and the need for human oversight.
"""

sample_questions = [
    "What industries or fields are mentioned?",
    "How can AI help in healthcare?",
    "What concerns does AI raise?",
]

print("Paragraph:")
print(sample_paragraph)

print("Summary:")
print(summarise_paragraph(sample_paragraph))

print("\nQuestion Answering Tests:")
for question in sample_questions:
    print(f"\nQuestion: {question}")
    print(f"Answer: {answer_question(sample_paragraph, question)}")

Paragraph:

Artificial intelligence is transforming many industries by helping organisations automate tasks,
analyse large amounts of data, and support decision making. In healthcare, AI systems can assist
with medical image analysis, patient monitoring, and drug discovery. In education, AI tools can
support personalised learning by adapting content to each student's needs. However, the use of AI
also raises concerns about privacy, fairness, transparency, and the need for human oversight.

Summary:
Artificial intelligence is transforming many industries by helping organisations automate tasks and analyse large amounts of data. In healthcare, AI systems can assist with medical image analysis, patient monitoring, and drug discovery. In education, AI tools cansupport personalised learning by adapting content to each student's needs.

Question Answering Tests:

Question: What industries or fields are mentioned?
Answer: healthcare

Question: How can AI help in healthcare?
Answer: medical im